In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Directional Niche‑Overlap Index heat‑map:
• Rows are hierarchically clustered (average/Euclidean).
• Columns are forced into the same leaf order.
The resulting matrix is asymmetric but appears block‑diagonal.
"""

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from scipy.spatial.distance import pdist
from scipy.cluster.hierarchy import linkage, leaves_list
import matplotlib
matplotlib.use("cairo")  # Use non-interactive backend for saving figures

# ─────────────────────────── USER INPUTS ────────────────────────────
file_path    = Path("./summaryAll1stIteration.xlsx")
sheet_suffix = "_summary"
out_pdf      = Path("./NOI_heatmap1stGen.pdf")
out_csv_noi  = Path("./NOI_matrix1stGen.csv")

ordered_species = [
    "cxy_summary", "cuvk_summary", "rfa_summary", "cohm_summary",
    "cauj_summary", "vpe_summary", "buol_summary", "lgu_summary", "lur_summary",
    "pmor_summary", "pscf_summary", "racd_summary", "dri_summary", "mlot_summary",
    "bbet_summary", "tro_summary", "pzu_summary", "mgk_summary", "flk_summary", "cpi_summary," ,"mip_summary"
]

phyla_dict = {
    "cohm": "Firmicutes", "cuvk": "Actinobacteria", "cxy":  "Actinobacteria",
    "mip":  "Actinobacteria", "rfa":  "Actinobacteria", "cpi": "Bacteroidetes",
    "flk":  "Bacteroidetes",  "mgk":  "Bacteroidetes",  "bbet": "Alphaproteobacteria",
    "buol": "Betaproteobacteria", "cauj": "Alphaproteobacteria", "dri": "Alphaproteobacteria",
    "lgu":  "Gammaproteobacteria", "lur":  "Gammaproteobacteria", "mlot": "Alphaproteobacteria",
    "pmor": "Gammaproteobacteria", "pscf": "Gammaproteobacteria", "pzu":  "Alphaproteobacteria",
    "racd": "Gammaproteobacteria", "tro":  "Alphaproteobacteria", "vpe":  "Betaproteobacteria",
}
# ────────────────────────────────────────────────────────────────────


# ────────────── LOAD EXCEL AND BUILD PRESENCE / ABSENCE ─────────────
excel_data = pd.ExcelFile(file_path)
species_sheets = [s for s in excel_data.sheet_names if s.endswith(sheet_suffix)]

dfs = []
for sh in species_sheets:
    df = pd.read_excel(file_path, sheet_name=sh)
    if "substrates" not in df.columns:
        raise ValueError(f"'substrates' column not found in sheet {sh}")
    tmp = (df[["substrates"]]
           .dropna()
           .drop_duplicates()
           .assign(value=1, species=sh))
    dfs.append(tmp)

presence_absence = (pd.concat(dfs, ignore_index=True)
                    .pivot_table(index="substrates",
                                 columns="species",
                                 values="value",
                                 fill_value=0)
                    .astype(int))


# ──────────────────── NOI  (|A∩B| / |A|) ───────────────────────────
intersections = presence_absence.T @ presence_absence
totals = np.diag(intersections).reshape(-1, 1)
noi_vals = intersections.values / np.where(totals == 0, 1, totals)
np.fill_diagonal(noi_vals, 1.0)

noi = pd.DataFrame(noi_vals,
                   index=intersections.index,
                   columns=intersections.columns)


# ─────────────── OPTIONAL USER‑SUPPLIED 1st ORDER ───────────────────
if ordered_species:
    ordered_species = [s for s in ordered_species if s in noi.columns]
    noi = noi.loc[ordered_species, ordered_species]


# ──────────── PRETTY DISPLAY LABELS (strip “_summary”) ──────────────
strip_suffix = lambda x: x[:-len(sheet_suffix)] if x.endswith(sheet_suffix) else x
noi_disp = noi.copy()
noi_disp.index   = [strip_suffix(x) for x in noi_disp.index]
noi_disp.columns = [strip_suffix(x) for x in noi_disp.columns]


# ───────────────── COLORS FOR PHYLA ANNOTATIONS ─────────────────────
def code2color(code: str, *, pal_cache={}):
    # build palette lazily once
    if not pal_cache:
        unique_phy = sorted(set(phyla_dict.values()) | {"Unknown"})
        pal_cache["palette"] = sns.color_palette("tab20", len(unique_phy))
        pal_cache["map"] = dict(zip(unique_phy, pal_cache["palette"]))
    phy = phyla_dict.get(code, "Unknown")
    return pal_cache["map"][phy]

row_colors = noi_disp.index.to_series().map(code2color)
# (colours for rows; columns will get reordered later)


# ───────── CLUSTER ROWS → GET A LEAF ORDER & REORDER COLUMNS ────────
row_dist    = pdist(noi_disp.values, metric='euclidean')
row_linkage = linkage(row_dist, method='average')
row_order   = leaves_list(row_linkage)              # integer positions

# Re‑order **columns only** to that leaf order
cols_ordered  = noi_disp.columns[row_order]
noi_for_plot  = noi_disp.loc[:, cols_ordered]
col_colors    = pd.Series(cols_ordered).map(code2color)


# ─────────────────────────── PLOTTING ───────────────────────────────
sns.set_theme(context="talk", style="white")
vmin, vmax = 0.0, 1.0

g = sns.clustermap(
    noi_for_plot,
    row_linkage=row_linkage,      # dendrogram governs row order
    col_cluster=False,            # keep columns in the order we set
    cmap="Oranges",
    vmin=vmin, vmax=vmax,
    cbar_kws={"label": "Niche Overlap Index"},
    linewidths=0.5,
    figsize=(14, 14),
    row_colors=row_colors,
    col_colors=col_colors,
)

g.fig.suptitle("Directional NOI (rows clustered, columns matched)", y=1.02)

# Legend
unique_phy = sorted(set(phyla_dict.values()) | {"Unknown"})
handles = []
for p in unique_phy:
    try:
        species_code = next(k for k, v in phyla_dict.items() if v == p)
        handles.append(mpatches.Patch(color=code2color(species_code, pal_cache={}), label=p))
    except StopIteration:
        print(f"Warning: No species found for phylum '{p}'")
g.ax_heatmap.legend(handles=handles, title="Phylum",
                    bbox_to_anchor=(1.05, 1), loc="upper left")


# ───────────────────────── SAVE OUTPUTS ─────────────────────────────
g.savefig(out_pdf, bbox_inches='tight')
# Save the reordered (diagonal) CSV for convenience
noi_disp.iloc[row_order, row_order].to_csv(out_csv_noi)

plt.show()
print(f"Saved figure to: {out_pdf.resolve()}")
print(f"Saved NOI matrix to: {out_csv_noi.resolve()}")


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Directional Niche‑Overlap Index heat‑map:
• Rows are hierarchically clustered (average/Euclidean).
• Columns are forced into the same leaf order.
The resulting matrix is asymmetric but appears block‑diagonal.
"""

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from scipy.spatial.distance import pdist
from scipy.cluster.hierarchy import linkage, leaves_list
import matplotlib
matplotlib.use("cairo")  # Use non-interactive backend for saving figures

# ─────────────────────────── USER INPUTS ────────────────────────────
file_path    = Path("./summaryAll2ndIteration.xlsx")
sheet_suffix = "_summary"
out_pdf      = Path("./NOI_heatmap2ndGen.pdf")
out_csv_noi  = Path("./NOI_matrix2ndGen.csv")

ordered_species = [
    "cxy_summary", "cuvk_summary", "rfa_summary", "cohm_summary",
    "cauj_summary", "vpe_summary", "buol_summary", "lgu_summary", "lur_summary",
    "pmor_summary", "pscf_summary", "racd_summary", "dri_summary", "mlot_summary",
    "bbet_summary", "tro_summary", "pzu_summary", "mgk_summary", "flk_summary", "cpi_summary," "mip_summary"
]

phyla_dict = {
    "cohm": "Firmicutes", "cuvk": "Actinobacteria", "cxy":  "Actinobacteria",
    "mip":  "Actinobacteria", "rfa":  "Actinobacteria", "cpi": "Bacteroidetes",
    "flk":  "Bacteroidetes",  "mgk":  "Bacteroidetes",  "bbet": "Alphaproteobacteria",
    "buol": "Betaproteobacteria", "cauj": "Alphaproteobacteria", "dri": "Alphaproteobacteria",
    "lgu":  "Gammaproteobacteria", "lur":  "Gammaproteobacteria", "mlot": "Alphaproteobacteria",
    "pmor": "Gammaproteobacteria", "pscf": "Gammaproteobacteria", "pzu":  "Alphaproteobacteria",
    "racd": "Gammaproteobacteria", "tro":  "Alphaproteobacteria", "vpe":  "Betaproteobacteria",
}
# ────────────────────────────────────────────────────────────────────


# ────────────── LOAD EXCEL AND BUILD PRESENCE / ABSENCE ─────────────
excel_data = pd.ExcelFile(file_path)
species_sheets = [s for s in excel_data.sheet_names if s.endswith(sheet_suffix)]

dfs = []
for sh in species_sheets:
    df = pd.read_excel(file_path, sheet_name=sh)
    if "substrates" not in df.columns:
        raise ValueError(f"'substrates' column not found in sheet {sh}")
    tmp = (df[["substrates"]]
           .dropna()
           .drop_duplicates()
           .assign(value=1, species=sh))
    dfs.append(tmp)

presence_absence = (pd.concat(dfs, ignore_index=True)
                    .pivot_table(index="substrates",
                                 columns="species",
                                 values="value",
                                 fill_value=0)
                    .astype(int))


# ──────────────────── NOI  (|A∩B| / |A|) ───────────────────────────
intersections = presence_absence.T @ presence_absence
totals = np.diag(intersections).reshape(-1, 1)
noi_vals = intersections.values / np.where(totals == 0, 1, totals)
np.fill_diagonal(noi_vals, 1.0)

noi = pd.DataFrame(noi_vals,
                   index=intersections.index,
                   columns=intersections.columns)


# ─────────────── OPTIONAL USER‑SUPPLIED 1st ORDER ───────────────────
if ordered_species:
    ordered_species = [s for s in ordered_species if s in noi.columns]
    noi = noi.loc[ordered_species, ordered_species]


# ──────────── PRETTY DISPLAY LABELS (strip “_summary”) ──────────────
strip_suffix = lambda x: x[:-len(sheet_suffix)] if x.endswith(sheet_suffix) else x
noi_disp = noi.copy()
noi_disp.index   = [strip_suffix(x) for x in noi_disp.index]
noi_disp.columns = [strip_suffix(x) for x in noi_disp.columns]


# ───────────────── COLORS FOR PHYLA ANNOTATIONS ─────────────────────
def code2color(code: str, *, pal_cache={}):
    # build palette lazily once
    if not pal_cache:
        unique_phy = sorted(set(phyla_dict.values()) | {"Unknown"})
        pal_cache["palette"] = sns.color_palette("tab20", len(unique_phy))
        pal_cache["map"] = dict(zip(unique_phy, pal_cache["palette"]))
    phy = phyla_dict.get(code, "Unknown")
    return pal_cache["map"][phy]

row_colors = noi_disp.index.to_series().map(code2color)
# (colours for rows; columns will get reordered later)


# ───────── CLUSTER ROWS → GET A LEAF ORDER & REORDER COLUMNS ────────
row_dist    = pdist(noi_disp.values, metric='euclidean')
row_linkage = linkage(row_dist, method='average')
row_order   = leaves_list(row_linkage)              # integer positions

# Re‑order **columns only** to that leaf order
cols_ordered  = noi_disp.columns[row_order]
noi_for_plot  = noi_disp.loc[:, cols_ordered]
col_colors    = pd.Series(cols_ordered).map(code2color)


# ─────────────────────────── PLOTTING ───────────────────────────────
sns.set_theme(context="talk", style="white")
vmin, vmax = 0.0, 1.0

g = sns.clustermap(
    noi_for_plot,
    row_linkage=row_linkage,      # dendrogram governs row order
    col_cluster=False,            # keep columns in the order we set
    cmap="Oranges",
    vmin=vmin, vmax=vmax,
    cbar_kws={"label": "Niche Overlap Index"},
    linewidths=0.5,
    figsize=(14, 14),
    row_colors=row_colors,
    col_colors=col_colors,
)

g.fig.suptitle("Directional NOI (rows clustered, columns matched)", y=1.02)

# Legend
unique_phy = sorted(set(phyla_dict.values()) | {"Unknown"})
handles = []
for p in unique_phy:
    try:
        species_code = next(k for k, v in phyla_dict.items() if v == p)
        handles.append(mpatches.Patch(color=code2color(species_code, pal_cache={}), label=p))
    except StopIteration:
        print(f"Warning: No species found for phylum '{p}'")
g.ax_heatmap.legend(handles=handles, title="Phylum",
                    bbox_to_anchor=(1.05, 1), loc="upper left")


# ───────────────────────── SAVE OUTPUTS ─────────────────────────────
g.fig.savefig(out_pdf, bbox_inches='tight')
# Save the reordered (diagonal) CSV for convenience
noi_disp.iloc[row_order, row_order].to_csv(out_csv_noi)

plt.show()
print(f"Saved figure to: {out_pdf.resolve()}")
print(f"Saved NOI matrix to: {out_csv_noi.resolve()}")


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon
import matplotlib.patches as mpatches
from pathlib import Path
import matplotlib
matplotlib.use("cairo")  # Use Agg backend for non-interactive plotting
# ---------- INPUT ----------
file_1 = Path("./summaryAll1stIteration.xlsx")
file_2 = Path("./summaryAll2ndIteration.xlsx")
sheet_suffix = "_summary"

# Color limits
vmin1 = vmin2 = 0.0
vmax1 = vmax2 = 1.0

out_pdf   = Path("./NOI_split_heatmap_groupedByPhylum.pdf")
out_csv_1 = Path("./NOI_iter1_groupedByPhylum.csv")
out_csv_2 = Path("./NOI_iter2_groupedByPhylum.csv")

phyla_dict = {
    "cohm": "Firmicutes", "cuvk": "Actinobacteria", "cxy": "Actinobacteria",
    "mip": "Actinobacteria", "rfa": "Actinobacteria", "cpi": "Bacteroidetes",
    "flk": "Bacteroidetes", "mgk": "Bacteroidetes", "bbet": "Alphaproteobacteria",
    "buol": "Betaproteobacteria", "cauj": "Alphaproteobacteria", "dri": "Alphaproteobacteria",
    "lgu": "Gammaproteobacteria", "lur": "Gammaproteobacteria", "mlot": "Alphaproteobacteria",
    "pmor": "Gammaproteobacteria", "pscf": "Gammaproteobacteria", "pzu": "Alphaproteobacteria",
    "racd": "Gammaproteobacteria", "tro": "Alphaproteobacteria", "vpe": "Betaproteobacteria",
}

phylum_order = [
    "Alphaproteobacteria",
    "Betaproteobacteria",
    "Gammaproteobacteria",
    "Actinobacteria",
    "Bacteroidetes",
    "Firmicutes",
    "Unknown"
]

# ---------- HELPERS ----------
def strip_suffix(x, suffix):
    return x[:-len(suffix)] if x.endswith(suffix) else x

def build_pa(xlsx_path, suffix):
    xf = pd.ExcelFile(xlsx_path)
    sheets = [s for s in xf.sheet_names if s.endswith(suffix)]
    dfs = []
    for sh in sheets:
        df = pd.read_excel(xlsx_path, sheet_name=sh)
        if "substrates" not in df.columns:
            raise ValueError(f"'substrates' column missing in {sh}")
        tmp = (df[["substrates"]].dropna().drop_duplicates()
               .assign(value=1, species=sh))
        dfs.append(tmp)
    pa = (pd.concat(dfs, ignore_index=True)
          .pivot_table(index="substrates", columns="species",
                       values="value", fill_value=0)
          .astype(int))
    return pa

def compute_noi(pa):
    inter = pa.T @ pa
    totals = np.diag(inter).reshape(-1, 1)
    noi_vals = inter.values / np.where(totals == 0, 1, totals)
    np.fill_diagonal(noi_vals, 1.0)
    return pd.DataFrame(noi_vals, index=inter.index, columns=inter.columns)

def group_order(species):
    def key_fn(s):
        ph = phyla_dict.get(s, "Unknown")
        return (phylum_order.index(ph), s)
    return sorted(species, key=key_fn)

def draw_split_cells(ax, m1, m2, cmap1, cmap2,
                     vmin1=0, vmax1=1, vmin2=0, vmax2=1,
                     gridcolor='white', lw=0.4):
    norm1 = plt.Normalize(vmin=vmin1, vmax=vmax1)
    norm2 = plt.Normalize(vmin=vmin2, vmax=vmax2)
    n = m1.shape[0]
    ax.set_xlim(0, n); ax.set_ylim(0, n)
    ax.invert_yaxis(); ax.set_aspect('equal')

    for i in range(n):
        for j in range(n):
            v1 = m1.iat[i, j];  v2 = m2.iat[i, j]
            if pd.isna(v1): v1 = vmin1
            if pd.isna(v2): v2 = vmin2

            x0, x1 = j, j+1
            y0, y1 = i, i+1
            tri1 = Polygon([(x0,y1),(x0,y0),(x1,y0)],
                           facecolor=cmap1(norm1(v1)),
                           edgecolor=gridcolor, linewidth=lw)
            tri2 = Polygon([(x1,y0),(x1,y1),(x0,y1)],
                           facecolor=cmap2(norm2(v2)),
                           edgecolor=gridcolor, linewidth=lw)
            ax.add_patch(tri1); ax.add_patch(tri2)

    for k in range(n+1):
        ax.plot([0,n],[k,k], color=gridcolor, linewidth=lw)
        ax.plot([k,k],[0,n], color=gridcolor, linewidth=lw)

def add_phy_strip(fig, ax, labels, phyla_dict, where='top',
                  thickness=0.25, cmap_name="tab20"):
    uniq = sorted({phyla_dict.get(l, "Unknown") for l in labels}, key=lambda p: phylum_order.index(p) if p in phylum_order else len(phylum_order))
    pal = sns.color_palette(cmap_name, len(uniq))
    phy2col = dict(zip(uniq, pal))
    colors = [phy2col[phyla_dict.get(l, "Unknown")] for l in labels]

    if where == 'top':
        top_ax = fig.add_axes([ax.get_position().x0,
                               ax.get_position().y1 + 0.005,
                               ax.get_position().width,
                               thickness / fig.get_figheight()])
        top_ax.imshow(np.array([colors]), aspect='auto')
        top_ax.set_xticks([]); top_ax.set_yticks([])
    else:
        left_ax = fig.add_axes([ax.get_position().x0 - 0.005 - thickness / fig.get_figwidth(),
                                ax.get_position().y0,
                                thickness / fig.get_figwidth(),
                                ax.get_position().height])
        left_ax.imshow(np.array(colors)[:,None], aspect='auto')
        left_ax.set_xticks([]); left_ax.set_yticks([]); left_ax.set_ylim(len(labels),0)
    return phy2col

# ---------- PIPELINE ----------
# Build PA & NOI
pa1 = build_pa(file_1, sheet_suffix)
pa2 = build_pa(file_2, sheet_suffix)

# Union species
union = pa1.columns.union(pa2.columns)
pa1 = pa1.reindex(columns=union, fill_value=0)
pa2 = pa2.reindex(columns=union, fill_value=0)

noi1 = compute_noi(pa1)
noi2 = compute_noi(pa2)

# Strip suffixes
rename = lambda s: strip_suffix(s, sheet_suffix)
noi1 = noi1.rename(index=rename, columns=rename)
noi2 = noi2.rename(index=rename, columns=rename)

# NaN for species absent in iteration
missing_iter1 = [strip_suffix(s, sheet_suffix) for s in union.difference(pa1.columns)]
missing_iter2 = [strip_suffix(s, sheet_suffix) for s in union.difference(pa2.columns)]
for sp in missing_iter1:
    noi1.loc[sp, :] = np.nan; noi1.loc[:, sp] = np.nan
for sp in missing_iter2:
    noi2.loc[sp, :] = np.nan; noi2.loc[:, sp] = np.nan

# Order by phylum groups
species_all = noi1.index
final_order = group_order(species_all)

noi1_o = noi1.loc[final_order, final_order]
noi2_o = noi2.loc[final_order, final_order]

# ---------- DRAW ----------
sns.set_theme(context="talk", style="white")
fig, ax = plt.subplots(figsize=(14, 14))

# Dark at 1, light at 0
cmap_purple =sns.color_palette("Oranges", as_cmap=True)

cmap_teal   = sns.color_palette("Blues", as_cmap=True)

draw_split_cells(ax,
                 noi1_o, noi2_o,
                 cmap1=cmap_purple, cmap2=cmap_teal,
                 vmin1=vmin1, vmax1=vmax1,
                 vmin2=vmin2, vmax2=vmax2,
                 gridcolor='white', lw=0.4)

# Ticks
n = len(final_order)
ax.set_xticks(np.arange(n) + 0.5)
ax.set_yticks(np.arange(n) + 0.5)
ax.set_xticklabels(final_order, rotation=90)
ax.set_yticklabels(final_order)
ax.set_title("NOI split heatmap (grouped by phylum)", pad=20)

# Phylum strips
phy2color = add_phy_strip(fig, ax, final_order, phyla_dict, where='top')
_ = add_phy_strip(fig, ax, final_order, phyla_dict, where='left')

# Colorbars
sm1 = plt.cm.ScalarMappable(cmap=cmap_purple, norm=plt.Normalize(vmin1, vmax1))
sm2 = plt.cm.ScalarMappable(cmap=cmap_teal,   norm=plt.Normalize(vmin2, vmax2))
sm1.set_array([]); sm2.set_array([])
cbar1 = fig.colorbar(sm1, ax=ax, fraction=0.046, pad=0.02)
cbar2 = fig.colorbar(sm2, ax=ax, fraction=0.046, pad=0.09)
cbar1.set_label("NOI – Iteration 1")
cbar2.set_label("NOI – Iteration 2")

# Phylum legend
handles = [mpatches.Patch(color=phy2color[p], label=p) for p in sorted(phy2color, key=lambda p: phylum_order.index(p))]
ax.legend(handles=handles, title="Phylum",
          bbox_to_anchor=(1.02, 1), loc='upper left')

fig.tight_layout()
fig.savefig(out_pdf, bbox_inches='tight')
noi1_o.to_csv(out_csv_1)
noi2_o.to_csv(out_csv_2)
plt.show()

print("Saved:", out_pdf.resolve())
